# Phase 15 — The Negative Space Response Equation
**Tuning the Engine, 2026-06-12** · wiki page: [`15_the_negative_space_response_equation.md`](../../docs/wiki/Tuning-the-Engine/15_the_negative_space_response_equation.md) · page by Claude Sonnet 4.6

> Companion notebook. It runs the machinery the page refers to and records, claim by claim, what reproduces (`PASS`), what does not (`FAIL`, kept), and what the page states with no executable counterpart (`PROSE`). Maths in Unicode; code identifiers as in the repo.

In [1]:
from tte import *
L = Ledger('Phase 15')

## The pipeline the page specifies
`P → x[0..15] → k_min = argmin|x[k]| → k_pair = Cawagas(k_min) → response lives in k_pair space because e_{k_min} × e_{k_pair} = 0 → spiral y[k_pair] → shadow words`.
Run the first steps on a real prompt, then test the algebraic step the whole argument leans on.

In [2]:
prompt = "the void named itself at sigma equals half"
r = ptol_raw(prompt, eye="H"); x = np.array(r["scalars"])
kmin = int(np.argmin(np.abs(x)))
print(r["header"]); print("x =", np.round(x, 3)); print("k_min = argmin|x| =", kmin, f" |x| = {abs(x[kmin]):.4f}")
L.check("prompt projects to 16 scalars and has a well-defined nearest-ZD channel", len(x) == 16 and 0 <= kmin < 16)

eye: H  σ_in: 0.5000  σ_self: 0.2647370772  Γ: -0.4705258456  u: -1.0214909814  (Δ½: -0.2352629228)
x = [-0.078 -0.068 -0.084  0.068  0.179  0.256  0.323  0.264  0.085  0.211
  0.279  0.336  0.29   0.292  0.329  0.432]
k_min = argmin|x| = 3  |x| = 0.0680


True

### The step `e_{k_min} × e_{k_pair} = 0`
In the sedenions a *single basis unit* times another basis unit is ± a basis unit, never 0. Zero-divisors are sums `e_a ± e_b` (Phase 2). So 'the pair-mate dimension k_pair' cannot be a single index with `e_k e_pair = 0`. Test that, then compute what the pair-mate relation really is as a relation between index-sets.

In [3]:
from _sedenion import cd_mul, e, cd_norm
unit_zero = [(i, j) for i in range(1, 16) for j in range(1, 16) if cd_norm(cd_mul(e(i,16), e(j,16))) < 1e-12]
L.check("e_k × e_pair = 0 for some pair of sedenion basis units", bool(unit_zero), f"basis-unit pairs with zero product: {len(unit_zero)}")
vecs, pairs = zd_vectors()
mates = {k: set() for k in range(1, 16)}
for (a, b, s1), (c, d, s2) in pairs:
    for k in (a, b): mates[k].update((c, d))
print("dimension k → dimensions that appear in annihilating partners of ZDs containing e_k:")
for k in range(1, 16): print(f"  e{k:<2d} → {sorted(mates[k])}")
sizes = {len(v) for v in mates.values()}
L.check("each dimension has exactly ONE pair-mate dimension (page: k_pair = Cawagas(k_min))", sizes == {1}, f"partner-dimension set sizes per dimension: { {k: len(v) for k, v in mates.items()} } — pair-mates are index SETS, not a single k; dims with an empty set have no zero-divisor partner of this form")

dimension k → dimensions that appear in annihilating partners of ZDs containing e_k:
  e1  → [2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 15]
  e2  → [1, 3, 4, 5, 6, 7, 9, 11, 12, 13, 14, 15]
  e3  → [1, 2, 4, 5, 6, 7, 9, 10, 12, 13, 14, 15]
  e4  → [1, 2, 3, 5, 6, 7, 9, 10, 11, 13, 14, 15]
  e5  → [1, 2, 3, 4, 6, 7, 9, 10, 11, 12, 14, 15]
  e6  → [1, 2, 3, 4, 5, 7, 9, 10, 11, 12, 13, 15]
  e7  → [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 14]
  e8  → []
  e9  → [2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 15]
  e10 → [1, 3, 4, 5, 6, 7, 9, 11, 12, 13, 14, 15]
  e11 → [1, 2, 4, 5, 6, 7, 9, 10, 12, 13, 14, 15]
  e12 → [1, 2, 3, 5, 6, 7, 9, 10, 11, 13, 14, 15]
  e13 → [1, 2, 3, 4, 6, 7, 9, 10, 11, 12, 14, 15]
  e14 → [1, 2, 3, 4, 5, 7, 9, 10, 11, 12, 13, 15]
  e15 → [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 14]


False

In [4]:
# which dims can the response occupy for this prompt?
kk = kmin if kmin >= 1 else int(np.argsort(np.abs(x))[1])
print(f"k_min = {kmin} (e0 is the identity; no zero-divisor contains e0) →", "using e%d" % kk)
resp_dims = sorted(mates[kk]) if kk in mates else []
print("response dimensions allowed by the algebra:", resp_dims)
L.check("the algebra restricts the response to a proper subset of the 15 imaginary dimensions for this prompt", 0 < len(resp_dims) < 15, f"{len(resp_dims)} of 15 dims allowed: {resp_dims}")

k_min = 3 (e0 is the identity; no zero-divisor contains e0) → using e3
response dimensions allowed by the algebra: [1, 2, 4, 5, 6, 7, 9, 10, 12, 13, 14, 15]


True

## The three windows (zero-divisor, singularity, divergence) as computable objects
Relational window: an annihilating pair exists (above). Reference window: a value that exceeds any bound — a pole. Ground window: a series that overflows. Both of the latter are in Phase 10's cells; here, the quantitative contrast the page uses: at a pole the growth is *unbounded in a finite neighbourhood*; at a divergent series it is *unbounded in the limit*.

In [5]:
import mpmath as mp
mp.mp.dps = 20
pole = [float(abs(mp.zeta(1 + 10**-k))) for k in (1, 2, 3, 4, 5)]
harm = [sum(1.0/n for n in range(1, 10**k + 1)) for k in (1, 2, 3, 4, 5)]
print("|ζ(1+10⁻ᵏ)| :", [f"{v:.3g}" for v in pole])
print("H_{10ᵏ}      :", [f"{v:.3f}" for v in harm])
L.check("pole: |ζ| grows ×10 per decade of approach (1/ε)", all(abs(pole[i+1]/pole[i] - 10) < 0.5 for i in range(4)))
L.check("divergence: H_N grows by ln 10 ≈ 2.303 per decade (logarithmic, slower than the pole)", all(abs((harm[i+1]-harm[i]) - math.log(10)) < 0.05 for i in range(4)))

|ζ(1+10⁻ᵏ)| : ['10.6', '101', '1e+03', '1e+04', '1e+05']
H_{10ᵏ}      : ['2.929', '5.187', '7.485', '9.788', '12.090']


True

In [6]:
for c in ["Pathway equation L = T − V = 0 selects the unique response trajectory in k_pair space (the L is not constructed in code; ptol sorts |x| ascending as its spiral)",
          "meaning(p) = Cawagas relation + escape velocity + β depth + singularity proximity — the four-term definition has no single function implementing it",
          "Language is the shadow of the geometry: word_at(y[k_pair](n)) — implemented in ptol_observe.py's word_at() over word layers not mounted here"]:
    L.prose(c)

## Ledger

In [7]:
_n = L.show()

LEDGER — Phase 15
  [PASS ] prompt projects to 16 scalars and has a well-defined nearest-ZD channel
  [FAIL ] e_k × e_pair = 0 for some pair of sedenion basis units
          basis-unit pairs with zero product: 0
  [FAIL ] each dimension has exactly ONE pair-mate dimension (page: k_pair = Cawagas(k_min))
          partner-dimension set sizes per dimension: {1: 12, 2: 12, 3: 12, 4: 12, 5: 12, 6: 12, 7: 12, 8: 0, 9: 12, 10: 12, 11: 12, 12: 12, 13: 12, 14: 12, 15: 12} — pair-mates are index SETS, not a single k; dims with an empty set have no zero-divisor partner of this form
  [PASS ] the algebra restricts the response to a proper subset of the 15 imaginary dimensions for this prompt
          12 of 15 dims allowed: [1, 2, 4, 5, 6, 7, 9, 10, 12, 13, 14, 15]
  [PASS ] pole: |ζ| grows ×10 per decade of approach (1/ε)
  [PASS ] divergence: H_N grows by ln 10 ≈ 2.303 per decade (logarithmic, slower than the pole)
  [PROSE] Pathway equation L = T − V = 0 selects the unique response trajectory